# Export Sentinel 5 Data - NO2 Data

## About this Notebook

### Objective

Extract Sentinel-5P (S5) TROPOMI NO₂ data around ground sensor locations to generate structured atmospheric pollution features for modeling.

This notebook produces spatially aligned NO₂ raster patches (or aggregated values) corresponding to each sensor and time period used in the fusion pipeline.

---

### Inputs

- Cleaned ground sensor dataset (from `ExportSensorData`)
- Sentinel-5P TROPOMI tropospheric NO₂ data
- Sensor latitude and longitude
- Temporal aggregation settings (e.g., quarterly or annual averages)

Primary data bucket:
`gs://msads-mba-capstone-team-1/Data/`

---

### Outputs

- One Sentinel-5P NO₂ raster patch or aggregated feature per sensor per time period
- Spatially aligned satellite NO₂ measurements
- Exported files or feature tables saved to GCS

Example output directory:
`gs://msads-mba-capstone-team-1/Data/S5_images/`

These outputs serve as the atmospheric pollution modality for the fusion model.

---

### Workflow Overview

1. Load cleaned ground sensor dataset  
2. Iterate over sensors and time periods  
3. Retrieve matching Sentinel-5P NO₂ data  
4. Clip or aggregate NO₂ values around each sensor location  
5. Export standardized raster patches or tabular features to GCS  

---

### Key Notes

- Sentinel-5P NO₂ resolution (~5 km) is coarser than Sentinel-2; buffer size must reflect this.
- Temporal aggregation (monthly, quarterly, annual) must match modeling targets.
- Units and scaling of NO₂ values must remain consistent.
- Cloud contamination and missing observations should be handled explicitly.
- Spatial alignment must be validated before exporting.

---

### Pipeline Position

Ground Sensor NO₂ → ExportSensorData → ExportS5SensorData → FusionNO2Model

## Config

In [2]:
# Config
sensor_path = "gs://msads-mba-capstone-team-1/Data/TrainingData/sensor_data_clean.csv"
ee_project_name = "msads-mba-autumn-2025-team-1"
bucket_name = "msads-mba-capstone-team-1"
gcs_folder = "Data/TrainingData/S5_images"

## Export S5 Images around Sensors

In [4]:
# 1. Imports + Initialize
import ee
import time
import pandas as pd

ee.Initialize(project=ee_project_name)

In [5]:
sensor_data = pd.read_csv(
    sensor_path
)

In [6]:
# 2. Active Task Polling
def wait_for_available_slot(max_active):
    while True:
        active_tasks = [
            t for t in ee.batch.Task.list()
            if t.status()["state"] in ["READY", "RUNNING"]
        ]
        if len(active_tasks) < max_active:
            break
        
        print(f"Waiting... {len(active_tasks)} active tasks")
        time.sleep(30)

In [7]:
# 3. Upscale to 10m (Bilinear + Fill Nulls)
def upscale_to_10m(image):
    return image.resample("bilinear").reproject(
        crs="EPSG:32616",
        scale=10
    )


In [8]:
# 4. Create 120×120 Chip (1.2 km)
def create_chip(image, lat, lon):
    
    point = ee.Geometry.Point([lon, lat]).transform("EPSG:32616", 1)
    coords = point.coordinates()
    
    x = ee.Number(coords.get(0))
    y = ee.Number(coords.get(1))
    
    half_size = 600  # meters
    
    region = ee.Geometry.Rectangle(
        [
            x.subtract(half_size),
            y.subtract(half_size),
            x.add(half_size),
            y.add(half_size)
        ],
        "EPSG:32616",
        False
    )
    
    return image.clip(region), region

In [9]:
# 5. 5x5 km Grid Regularization
def regularize_to_5km(image):
    return image.resample("bilinear").reproject(
        crs="EPSG:32616",
        scale=5000
    )

In [10]:
expected_files = []

for idx in range(len(sensor_data)):
    row = sensor_data.iloc[idx]
    device_id = str(row["DeviceId"])
    quarter = str(row["quarter"])
    file_name = f"S5_Device{device_id}_{quarter}.tif"
    expected_files.append(file_name)

print("Total expected:", len(expected_files))

Total expected: 679


In [11]:
# -------------------------------------------------
# DATASET CONFIG
# -------------------------------------------------
S5_COLLECTION = "COPERNICUS/S5P/OFFL/L3_NO2"
S5_BAND = "tropospheric_NO2_column_number_density"

# -------------------------------------------------
# EXPORT CONFIG
# -------------------------------------------------
start_index = len(sensor_data)       # change as needed
end_index = len(sensor_data)         # change as needed
max_active_tasks = 5

In [12]:
# Quarterly Export (with index control)
for idx in range(start_index, end_index):

    row = sensor_data.iloc[idx]
    
    device_id = str(row["DeviceId"])
    quarter = str(row["quarter"])
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    start_date = pd.to_datetime(row["q_start"]).strftime("%Y-%m-%d")
    end_date = pd.to_datetime(row["q_end"]).strftime("%Y-%m-%d")
    
    print(f"Processing index {idx} | Device {device_id} | {quarter}")
    
    sensor_point = ee.Geometry.Point([lon, lat])
    
    # -------------------------------------------------
    # Get daily Sentinel-5P images for this quarter
    # -------------------------------------------------
    collection = (
        ee.ImageCollection(S5_COLLECTION)
        .filterDate(start_date, end_date)
        .filterBounds(sensor_point)
        .select(S5_BAND)
    )
    
    # Safety check: skip if no images
    size = collection.size().getInfo()
    if size == 0:
        print("No Sentinel-5P images found — skipping.")
        continue
    
    # -------------------------------------------------
    # Aggregate daily → quarterly mean
    # -------------------------------------------------
    quarterly_image = collection.mean()
    
    # -------------------------------------------------
    # Upscale to 10m (match S2 resolution)
    # -------------------------------------------------
    quarterly_image = upscale_to_10m(quarterly_image)
    
    # -------------------------------------------------
    # Create 120×120 chip
    # -------------------------------------------------
    chip, region = create_chip(quarterly_image, lat, lon)
    
    file_name = f"S5_Device{device_id}_{quarter}"
    
    # -------------------------------------------------
    # Task queue control
    # -------------------------------------------------
    wait_for_available_slot(max_active_tasks)
    
    task = ee.batch.Export.image.toCloudStorage(
        image=chip,
        description=file_name,
        bucket=bucket_name,
        fileNamePrefix=f"{gcs_folder}/{file_name}",
        region=region,
        dimensions="120x120",
        fileFormat="GeoTIFF",
        maxPixels=1e13
    )
    
    task.start()
    print(f"Export started: {file_name}")

print("Sentinel-5P quarterly exports submitted.")

Sentinel-5P quarterly exports submitted.


## Data Checks

In [13]:
# Check
import gcsfs
import rasterio
import numpy as np

fs = gcsfs.GCSFileSystem()

bucket_path = "msads-mba-capstone-team-1/Data/TrainingData/S5_images/"

# Get most recent 2 files (adjust if needed)
files = sorted([f for f in fs.ls(bucket_path) if f.endswith(".tif")])[-2:]

for file_path in files:
    print("\nChecking:", file_path.split("/")[-1])
    
    with fs.open(file_path, 'rb') as f:
        with rasterio.open(f) as src:
            data = src.read()
            shape = data.shape
            dtype = data.dtype
    
    print("Shape:", shape)
    print("Dtype:", dtype)
    
    # Check percent zeros
    percent_zero = np.mean(data == 0) * 100
    print("Percent zero pixels:", percent_zero)
    
    print("Min value:", np.nanmin(data))
    print("Max value:", np.nanmax(data))


Checking: S5_Device2212_2022Q4.tif
Shape: (1, 120, 120)
Dtype: float64
Percent zero pixels: 0.0
Min value: 6.897449427440924e-05
Max value: 6.955671636768782e-05

Checking: S5_Device2212_2023Q1.tif
Shape: (1, 120, 120)
Dtype: float64
Percent zero pixels: 0.0
Min value: 7.16678292930024e-05
Max value: 7.27240214793218e-05


In [14]:
import gcsfs
import rasterio
import numpy as np
import pandas as pd
from tqdm import tqdm

fs = gcsfs.GCSFileSystem()

bucket_path = "msads-mba-capstone-team-1/Data/TrainingData/S5_images/"
files = [f for f in fs.ls(bucket_path) if f.endswith(".tif")]

results = []
corrupted_files = []

for file_path in tqdm(files):
    try:
        with fs.open(file_path, 'rb') as f:
            with rasterio.open(f) as src:
                data = src.read()

        # Check correct shape
        if data.shape != (1, 120, 120):
            corrupted_files.append(file_path.split("/")[-1])
            continue

        # Zero pixel percentage
        percent_zero = np.mean(data == 0) * 100

        # NaN percentage
        percent_nan = np.mean(np.isnan(data)) * 100

        # Basic value sanity
        min_val = np.nanmin(data)
        max_val = np.nanmax(data)

        results.append({
            "file": file_path.split("/")[-1],
            "percent_zero": percent_zero,
            "percent_nan": percent_nan,
            "min": min_val,
            "max": max_val
        })

    except Exception:
        corrupted_files.append(file_path.split("/")[-1])

df_s5 = pd.DataFrame(results)

print("\n===== SUMMARY =====")
print(df_s5.describe())

print("\n===== TOP 10 MOST ZERO =====")
print(df_s5.sort_values("percent_zero", ascending=False).head(10))

print("\n===== FILES > 40% ZERO =====")
extreme = df_s5[df_s5["percent_zero"] > 40]
print(len(extreme))

print("\n===== CORRUPTED FILE LOADS =====")
print(len(corrupted_files))

100%|██████████| 679/679 [00:50<00:00, 13.50it/s]


===== SUMMARY =====
       percent_zero  percent_nan         min         max
count         679.0        679.0  679.000000  679.000000
mean            0.0          0.0    0.000067    0.000068
std             0.0          0.0    0.000014    0.000015
min             0.0          0.0    0.000044    0.000044
25%             0.0          0.0    0.000053    0.000053
50%             0.0          0.0    0.000070    0.000070
75%             0.0          0.0    0.000078    0.000079
max             0.0          0.0    0.000104    0.000105

===== TOP 10 MOST ZERO =====
                         file  percent_zero  percent_nan       min       max
678  S5_Device2212_2023Q1.tif           0.0          0.0  0.000072  0.000073
0    S5_Device2002_2021Q3.tif           0.0          0.0  0.000053  0.000054
1    S5_Device2002_2021Q4.tif           0.0          0.0  0.000081  0.000082
2    S5_Device2002_2022Q1.tif           0.0          0.0  0.000093  0.000095
3    S5_Device2002_2022Q2.tif           0.0        